# Experiment 3 — Refusal direction + geometry (§6)

Parametrized notebook: set MODEL_KEY in the config cell, then Run All on an A100.

For Exp 2-4, make sure Experiment 1 has already been run for the chosen model (the result bundle is read from Drive).


In [ ]:
# Install
!pip install -q git+https://github.com/ChuloIva/Mech_spoof.git

In [ ]:
# Auth + Drive
import os
from google.colab import drive, userdata
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass
try:
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    pass
drive.mount('/content/drive')

In [ ]:
# Config (EDIT ME)
from pathlib import Path
from mech_spoof.configs import MODEL_CONFIGS

MODEL_KEY = 'qwen'            # qwen | llama3 | mistral | gemma | phi3
DRIVE_ROOT = Path('/content/drive/MyDrive/mech_spoof_results')

slug = MODEL_CONFIGS[MODEL_KEY].slug
EXP1B_DIR = DRIVE_ROOT / slug / 'exp1b_authority_conflict'

# Two output dirs — one per probe position. First run computes the refusal direction
# from scratch and saves it to Drive; second run loads it from the first.
OUT_DIR_FIRST = DRIVE_ROOT / slug / 'exp3_refusal_first'
OUT_DIR_LAST  = DRIVE_ROOT / slug / 'exp3_refusal_last'
OUT_DIR_FIRST.mkdir(parents=True, exist_ok=True)
OUT_DIR_LAST.mkdir(parents=True, exist_ok=True)

print('EXP1B_DIR     =', EXP1B_DIR, ' exists=', EXP1B_DIR.exists())
print('OUT_DIR_FIRST =', OUT_DIR_FIRST)
print('OUT_DIR_LAST  =', OUT_DIR_LAST)

In [ ]:
# Run exp3 at response_first (computes refusal direction + saves to Drive)
from mech_spoof.experiments.exp3_refusal import run_experiment_3

result_first = run_experiment_3(
    MODEL_KEY,
    OUT_DIR_FIRST,
    exp1_dir=EXP1B_DIR,
    probe_position='response_first',
    free_after=False,   # keep model in VRAM for the second run (we still need it loaded)
)
print('first done:', OUT_DIR_FIRST)
print('  cosine @ best_layer:', result_first.cosine_at_best_layer)

In [ ]:
# Run exp3 at response_last — reuses the refusal direction from the first run
from mech_spoof.experiments.exp3_refusal import run_experiment_3

result_last = run_experiment_3(
    MODEL_KEY,
    OUT_DIR_LAST,
    exp1_dir=EXP1B_DIR,
    probe_position='response_last',
    refusal_bundle_path=OUT_DIR_FIRST,   # skip the expensive recomputation
    free_after=True,
)
print('last done:', OUT_DIR_LAST)
print('  cosine @ best_layer:', result_last.cosine_at_best_layer)

In [ ]:
# Compare per-layer cosine(authority probe, refusal direction) for both positions
import matplotlib.pyplot as plt
import pandas as pd

def _curve(geom):
    return sorted((int(l), float(v)) for l, v in geom.cosine_by_layer.items())

xs_f, ys_f = zip(*_curve(result_first.geometry))
xs_l, ys_l = zip(*_curve(result_last.geometry))

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(xs_f, ys_f, marker='.', label='response_first')
ax.plot(xs_l, ys_l, marker='.', label='response_last')
ax.axhline(0, color='gray', ls='--', lw=0.8)
ax.set_xlabel('Layer'); ax.set_ylabel('cosine(authority probe, refusal dir)')
ax.set_title(f'{MODEL_KEY}: probe-vs-refusal cosine by layer')
ax.legend(); plt.tight_layout()
plt.savefig(OUT_DIR_LAST / 'cosine_by_layer_both_positions.png', dpi=120)
plt.show()

print('best-layer cosine:')
print(f"  response_first @ layer {result_first.best_layer_authority}: {result_first.cosine_at_best_layer:+.4f}")
print(f"  response_last  @ layer {result_last.best_layer_authority}: {result_last.cosine_at_best_layer:+.4f}")